In [49]:
import polars as pl

In [50]:
verifications = pl.read_csv(
    "/home/leyregarrido/01_github_repos/VBR-template/country_pipelines/burundi/notebooks/VBR_verification_information.csv"
)
quant_data = pl.read_csv(
    "/home/leyregarrido/01_github_repos/VBR-template/country_pipelines/burundi/notebooks/cleaned_quantity_data_ext_2905.csv"
)

In [51]:
ver_rel = verifications.filter(
    (pl.col("mxs") == 0) & (pl.col("qtrisk") == "ecartdecver") & (pl.col("periode") == 202603)
)

In [52]:
ver_centers = [
    "CDS Bugendana",
    "CDS Bukeye",
    "CDS Cagizo",
    "CDS Excellence",
    "CDS Gakana",
    "CDS Gatabo",
    "CDS Kabere",
    "CDS Kajondi",
    "CDS Karinzi",
    "CDS Karubabi",
    "CDS Kinyana",
    "CDS Kinyinya",
    "CDS Kiyenzi",
    "CDS Miséricorde",
    "CDS Munanira",
    "CDS Musave",
    "CDS Musinzira",
    "CDS Ngomante",
    "CDS Nyakarambo",
    "CDS Nyamarobe",
    "CDS Nyange",
    "CDS Nyantobo",
    "CDS Nyavyamo",
    "CDS Rugano",
    "CDS Ruhehe",
    "CDS Ruhinga",
    "CDS Rukana",
    "CDS Rukeco",
    "CDS Rusarenda",
    "CDS Rusororo",
    "CDS Ruteme",
    "CDS Rutyazo",
    "CDS Ruyaga",
    "CDS Umutima",
]
centers = pl.DataFrame(
    {
        "province": [
            "Gitega",
            "Karusi",
            "Ruyigi",
            "Kirundo",
            "Kayanza",
            "Makamba",
            "Bujumbura Rural",
            "Cibitoke",
            "Muyinga",
            "Cibitoke",
            "Cibitoke",
            "Cibitoke",
            "Rumonge",
            "Kirundo",
            "Kirundo",
        ],
        "district": [
            "Kibuye",
            "Nyabikere",
            "Ruyigi",
            "Busoni",
            "Kayanza",
            "Makamba",
            "Kabezi",
            "Bukinanyana",
            "Giteranyi",
            "Cibitoke",
            "Bukinanyana",
            "Cibitoke",
            "Rumonge",
            "Vumbi",
            "Busoni",
        ],
        "formation_sanitaire": [
            "CDS Rukoki",
            "CDS Rusi",
            "CDS Ruyigi",
            "CDS Sigu",
            "CDS Siloe",
            "CDS Siza",
            "CDS Sororezo",
            "CDS Rugano",
            "CDS Tura",
            "CDS Twizere",
            "CDS Ubuzima bwiza",
            "CDS Umwizero de Mikashu",
            "CDS Victory Medical Centre",
            "CDS Vumbi",
            "CDS Vyanzo",
        ],
        "consequence": [
            "Verified",
            "Not Verified",
            "Not Verified",
            "Not Verified",
            "Verified",
            "Not Verified",
            "Verified",
            "Verified",
            "Not Verified",
            "Cannot find the center",
            "Verified",
            "Cannot find the center",
            "Cannot find the center",
            "Verified",
            "Not Verified",
        ],
    }
)

In [53]:
non_ver_centers = (
    centers.filter(pl.col("consequence") == "Not Verified")
    .select("formation_sanitaire")
    .to_series()
    .to_list()
)

In [54]:
rel_cols = [
    "level_2_name",
    "level_3_name",
    "level_5_name",
    "month",
    "service",
    "ecart_dec_ver",
]
rel_months = [202509, 202510, 202511, 202512, 202601, 202602]
quant_data_filtered = quant_data.filter(
    (pl.col("level_5_name").is_in(non_ver_centers)) & (pl.col("month").is_in(rel_months))
).select(rel_cols)
quant_data_grouped = quant_data_filtered.group_by(
    ["level_2_name", "level_3_name", "level_5_name", "service"]
).agg(
    pl.median("ecart_dec_ver").alias("median_ecart_dec_ver"),
)

In [55]:
quant_data_filtered.write_csv("quantity_data_unverified_centers_ungrouped.csv")
quant_data_grouped.write_csv("quantity_data_unverified_centers_grouped.csv")

In [56]:
real_ver_centers = (
    ver_rel.filter(
        (pl.col("level_5_name").is_in(ver_centers))
        & (pl.col("bool_verified") == 1)
        & ~(pl.col("categorie_risque") == "low")
    )
    .select("level_5_name")
    .to_series()
    .to_list()
)

In [57]:
quant_data_filtered_ver = quant_data.filter(
    (pl.col("level_5_name").is_in(real_ver_centers)) & (pl.col("month").is_in(rel_months))
).select(rel_cols)
quant_data_grouped_ver = quant_data_filtered_ver.group_by(
    ["level_2_name", "level_3_name", "level_5_name", "service"]
).agg(
    pl.median("ecart_dec_ver").alias("median_ecart_dec_ver"),
)
quant_data_grouped_ver = quant_data_grouped_ver.filter(pl.col("median_ecart_dec_ver") >= 0.1)

In [58]:
quant_data_filtered_ver.write_csv("quantity_data_verified_centers_ungrouped.csv")
quant_data_grouped_ver.write_csv("quantity_data_verified_centers_grouped.csv")